In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, average_precision_score, f1_score, 
    precision_recall_curve, roc_curve, roc_auc_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)
from codecarbon import EmissionsTracker

# ====== CONFIG ======
IN_FILE = "risultati_merged_mixed.csv"
MODEL_OUT = "catboost_td_distilled.cbm"
META_OUT = "catboost_td_distilled_meta.json"
PLOTS_DIR = "plots_catboost"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"
LABEL_COL = "label"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Crea directory per i grafici
os.makedirs(PLOTS_DIR, exist_ok=True)

# ====== LOAD ======
print("Caricamento dataset...")
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

print(f"Dataset caricato: {len(df)} righe")

# ====== FEATURE COLUMNS ======
num_cols = [c for c in df.columns if c.startswith("sal_") or c.startswith("n_tokens_")]
num_cols = [c for c in num_cols if c not in [TARGET_COL, LABEL_COL]]
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

for c in num_cols:
    if (
        c.startswith("n_tokens_")
        or c.endswith("_sum")
        or c.endswith("_cnt")
        or c.endswith("_pct")
        or c.endswith("_share")
    ):
        df[c] = df[c].fillna(0.0)
    else:
        df[c] = df[c].fillna(df[c].median())

feature_cols = [TEXT_COL]

# ====== SPLIT ======
print("Split train/validation...")
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

print(f"Train: {len(train_df)} | Validation: {len(valid_df)}")

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN CON CARBON TRACKING ======
print("\nInizio training con carbon tracking...")
tracker = EmissionsTracker(project_name="catboost_td", output_dir=PLOTS_DIR)
tracker.start()

start_time = time.time()

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

training_time = time.time() - start_time
emissions = tracker.stop()

print(f"\nTraining completato in {training_time:.2f} secondi ({training_time/60:.2f} minuti)")
print(f"Emissioni CO2: {emissions:.6f} kg")

# ====== PREDICTIONS ======
print("\nGenerazione predizioni...")
pred_logit_train = model.predict(train_pool)
pred_logit = model.predict(valid_pool)

# ====== EVAL (distillation fidelity) ======
rmse_train = np.sqrt(mean_squared_error(train_df[TARGET_COL].values, pred_logit_train))
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) train: {rmse_train:.6f}")
print(f"RMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (classification vs label) ======
p_hat_train = 1.0 / (1.0 + np.exp(-pred_logit_train))
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))

metrics = {}

if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_train = train_df[LABEL_COL].astype(int).values
    y_true = valid_df[LABEL_COL].astype(int).values
    
    # ROC-AUC
    roc_auc_train = roc_auc_score(y_train, p_hat_train)
    roc_auc = roc_auc_score(y_true, p_hat)
    print(f"\nROC-AUC train: {roc_auc_train:.6f}")
    print(f"ROC-AUC valid: {roc_auc:.6f}")
    
    # PR-AUC
    pr_auc_train = average_precision_score(y_train, p_hat_train)
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC train: {pr_auc_train:.6f}")
    print(f"PR-AUC valid: {pr_auc:.6f}")
    
    # Trova soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    
    y_pred = (p_hat >= best_thr).astype(int)
    
    # Metriche di classificazione
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\nSoglia ottimale (F1): {best_thr:.6f}")
    print(f"Accuracy: {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall: {recall:.6f}")
    print(f"F1-Score: {f1:.6f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    metrics = {
        "roc_auc_train": roc_auc_train,
        "roc_auc_valid": roc_auc,
        "pr_auc_train": pr_auc_train,
        "pr_auc_valid": pr_auc,
        "best_threshold": best_thr,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }
    
    # ====== PLOT ROC CURVE ======
    fpr, tpr, _ = roc_curve(y_true, p_hat)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - CatBoost TD Classifier', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "roc_curve.png"), dpi=300)
    print(f"\nROC curve salvata in: {os.path.join(PLOTS_DIR, 'roc_curve.png')}")
    plt.close()
    
    # ====== PLOT PR CURVE ======
    plt.figure(figsize=(8, 6))
    plt.plot(recalls, precisions, linewidth=2, label=f'PR curve (AUC = {pr_auc:.4f})')
    plt.axhline(y=y_true.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline ({y_true.mean():.4f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curve - CatBoost TD Classifier', fontsize=14, fontweight='bold')
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "pr_curve.png"), dpi=300)
    print(f"PR curve salvata in: {os.path.join(PLOTS_DIR, 'pr_curve.png')}")
    plt.close()
    
    # ====== PLOT CONFUSION MATRIX ======
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Non-TD', 'TD'], yticklabels=['Non-TD', 'TD'])
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix - CatBoost TD Classifier', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "confusion_matrix.png"), dpi=300)
    print(f"Confusion matrix salvata in: {os.path.join(PLOTS_DIR, 'confusion_matrix.png')}")
    plt.close()
    
    # ====== PLOT DISTRIBUTION OF PREDICTIONS ======
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribuzione per classe vera
    ax1.hist(p_hat[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax1.hist(p_hat[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax1.axvline(best_thr, color='green', linestyle='--', linewidth=2, label=f'Soglia ottimale ({best_thr:.3f})')
    ax1.set_xlabel('Predicted Probability', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Logit distribution
    ax2.hist(pred_logit[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax2.hist(pred_logit[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax2.set_xlabel('Predicted Logit', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Distribution of Predicted Logits', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "prediction_distributions.png"), dpi=300)
    print(f"Distribution plots salvate in: {os.path.join(PLOTS_DIR, 'prediction_distributions.png')}")
    plt.close()
    
else:
    print("Label non presente -> salto metriche classificazione.")

# ====== FEATURE IMPORTANCE ======
try:
    # CatBoost con text features mantiene gli indici
    feature_importance = model.get_feature_importance(train_pool)
    feature_names = feature_cols
    
    if len(feature_importance) > 0:
        # Crea dataframe con importanze
        fi_df = pd.DataFrame({
            'feature': feature_names[:len(feature_importance)],
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print("\nTop 20 Feature Importance:")
        print(fi_df.head(20).to_string(index=False))
        
        # Plot top features
        top_n = min(20, len(fi_df))
        plt.figure(figsize=(10, max(6, top_n * 0.3)))
        plt.barh(range(top_n), fi_df['importance'].head(top_n), color='steelblue')
        plt.yticks(range(top_n), fi_df['feature'].head(top_n))
        plt.xlabel('Importance', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.title(f'Top {top_n} Feature Importance - CatBoost', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "feature_importance.png"), dpi=300)
        print(f"\nFeature importance plot salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.png')}")
        plt.close()
        
        # Salva feature importance in CSV
        fi_df.to_csv(os.path.join(PLOTS_DIR, "feature_importance.csv"), index=False)
        print(f"Feature importance CSV salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.csv')}")
except Exception as e:
    print(f"\nNote: Impossibile calcolare feature importance: {e}")

# ====== SAVE MODEL ======
model.save_model(MODEL_OUT)

# ====== SAVE METADATA ======
meta = {
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df.columns else None,
    "num_cols": num_cols,
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "dataset_size": len(df),
    "train_size": len(train_df),
    "valid_size": len(valid_df),
    "rmse_train_logit": float(rmse_train),
    "rmse_valid_logit": float(rmse),
    "training_time_seconds": training_time,
    "training_time_minutes": training_time / 60,
    "co2_emissions_kg": float(emissions),
    "model_iterations": model.get_best_iteration(),
    "model_file": MODEL_OUT,
    **metrics
}

with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("RIEPILOGO FINALE")
print("="*60)
print(f"Modello salvato: {MODEL_OUT}")
print(f"Metadata salvati: {META_OUT}")
print(f"Grafici salvati in: {PLOTS_DIR}/")
print(f"Numero features: {len(feature_cols)}")
print(f"Tempo training: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"CO2 emissions: {emissions:.6f} kg")
print(f"RMSE validation: {rmse:.6f}")
if metrics:
    print(f"ROC-AUC validation: {metrics.get('roc_auc_valid', 'N/A'):.6f}")
    print(f"PR-AUC validation: {metrics.get('pr_auc_valid', 'N/A'):.6f}")
    print(f"F1-Score validation: {metrics.get('f1_score', 'N/A'):.6f}")
print("="*60)


Caricamento dataset...


[codecarbon WARNING @ 19:51:41] Multiple instances of codecarbon are allowed to run at the same time.


Dataset caricato: 20828 righe
Split train/validation...
Train: 16662 | Validation: 4166

Inizio training con carbon tracking...


[codecarbon INFO @ 19:51:46] [setup] RAM Tracking...
[codecarbon INFO @ 19:51:46] [setup] CPU Tracking...
[codecarbon WARNING @ 19:51:48] We saw that you have a 13th Gen Intel(R) Core(TM) i7-13620H but we don't know it. Please contact us.
[codecarbon WARNING @ 19:51:48] We will use the default power consumption of 4 W per thread for your 16 CPU, so 64W.
[codecarbon WARNING @ 19:51:48] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 19:51:48] CPU Model on constant consumption mode: 13th Gen Intel(R) Core(TM) i7-13620H
[codecarbon WARNING @ 19:51:48] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 19:51:48] [setup] GPU Tracking...
[codecarbon INFO @ 19:51:48] No GPU found.
[codecarbon INFO @ 19:51:48] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Me

0:	learn: 4.0431939	test: 4.0426953	best: 4.0426953 (0)	total: 227ms	remaining: 18m 54s


...

2600:	learn: 1.5804625	test: 2.4892764	best: 2.4884595 (2453)	total: 5m 27s	remaining: 5m 2s


[codecarbon INFO @ 19:57:24] Delta energy consumed for CPU with cpu_load : 0.000144 kWh, power : 35.627790323199996 W
[codecarbon INFO @ 19:57:24] Energy consumed for All CPU : 0.003127 kWh
[codecarbon INFO @ 19:57:24] 0.004017 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 19:57:30] Energy consumed for RAM : 0.000909 kWh. RAM Power : 10.0 W


Stopped by overfitting detector  (200 iterations wait)

bestTest = 2.488459497
bestIteration = 2453

Shrink model to first 2454 iterations.


[codecarbon INFO @ 19:57:31] Delta energy consumed for CPU with cpu_load : 0.000062 kWh, power : 33.127744456 W
[codecarbon INFO @ 19:57:31] Energy consumed for All CPU : 0.003189 kWh
[codecarbon INFO @ 19:57:31] 0.004098 kWh of electricity and 0.000000 L of water were used since the beginning.



Training completato in 337.67 secondi (5.63 minuti)
Emissioni CO2: 0.001355 kg

Generazione predizioni...

RMSE(logit) train: 1.613342
RMSE(logit) valid: 2.488459

ROC-AUC train: 0.981345
ROC-AUC valid: 0.938684
PR-AUC train: 0.982121
PR-AUC valid: 0.942744

Soglia ottimale (F1): 0.394784
Accuracy: 0.862698
Precision: 0.868563
Recall: 0.853494
F1-Score: 0.860963

Confusion Matrix:
[[1823  268]
 [ 304 1771]]

ROC curve salvata in: plots_catboost\roc_curve.png
PR curve salvata in: plots_catboost\pr_curve.png
Confusion matrix salvata in: plots_catboost\confusion_matrix.png
Distribution plots salvate in: plots_catboost\prediction_distributions.png

Top 20 Feature Importance:
  feature  importance
orig_text       100.0

Feature importance plot salvato in: plots_catboost\feature_importance.png
Feature importance CSV salvato in: plots_catboost\feature_importance.csv

RIEPILOGO FINALE
Modello salvato: catboost_td_distilled.cbm
Metadata salvati: catboost_td_distilled_meta.json
Grafici salvati 